本模块为探索性数据分析（EDA）阶段，主要目的是产出论文第五节所需的全部基础统计口径表格。通过聚合计算总体经营概览、月度走势、各类目与区域的结构占比、客群特征、折扣档位以及头部与亏损商品等维度，生成一系列可引用的明细表。该阶段不仅为后续的深度商业建模提供数据参考，也为描述性统计和异常值判断奠定基础。

In [1]:
import os
import numpy as np
import pandas as pd

current_dir = os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, ".."))
CLEAN_DIR = os.path.join(base_dir, "data", "clean")
TAB_DIR = os.path.join(base_dir, "outputs", "tables")
os.makedirs(TAB_DIR, exist_ok=True)

def load():
    return pd.read_csv(os.path.join(CLEAN_DIR, "orders_clean.csv"), parse_dates=["Order Date", "Ship Date"])

def dump(df, name):
    p = os.path.join(TAB_DIR, f"{name}.csv")
    df.to_csv(p, index=False, encoding="utf-8-sig")
    print(f"   {name}.csv  {df.shape[0]} 行")

print("探索性分析")
df = load()

sales, profit = df["Sales"].sum(), df["Profit"].sum()
loss = df[df["Is Loss"]]
t = pd.DataFrame([
    ("销售总额", f"{sales/1e4:,.1f} 万美元"), ("利润总额", f"{profit/1e4:,.1f} 万美元"),
    ("综合利润率", f"{profit/sales*100:.2f}%"), ("订单数", f"{df['Order ID'].nunique():,}"),
    ("客户数", f"{df['Customer ID'].nunique():,}"), ("商品数", f"{df['Product ID'].nunique():,}"),
    ("明细行数", f"{len(df):,}"), ("亏损行数占比", f"{len(loss)/len(df)*100:.1f}%（{len(loss):,} 行）"),
    ("亏损金额", f"{loss['Profit'].sum()/1e4:,.1f} 万美元"), ("亏损行销售额占比", f"{loss['Sales'].sum()/sales*100:.1f}%"),
    ("客单价（AOV）", f"{sales/df['Order ID'].nunique():,.0f} 美元"), ("人均消费（ARPU）", f"{sales/df['Customer ID'].nunique():,.0f} 美元"),
    ("平均折扣率", f"{df['Discount'].mean()*100:.2f}%"), ("有折扣订单占比", f"{(df.groupby('Order ID')['Discount'].max()>0).mean()*100:.1f}%"),
    ("平均发货时效", f"{df['Ship Days'].mean():.2f} 天"),
], columns=["指标", "取值"])
dump(t, "t10_overview")

m = (df.groupby("Order YM").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"), Qty=("Quantity", "sum")).reset_index())
m["Margin"] = m["Profit"] / m["Sales"]
m["MoM_Sales"] = m["Sales"].pct_change()
m["YoY_Sales"] = m["Sales"].pct_change(12)
m["MA3_Sales"] = m["Sales"].rolling(3, center=True).mean()
m["MA3_Profit"] = m["Profit"].rolling(3, center=True).mean()
dump(m.round(4), "t11_monthly")

def agg_by(keys):
    g = (df.groupby(keys).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Qty=("Quantity", "sum"), Rows=("Sales", "size"), Orders=("Order ID", "nunique")).reset_index())
    g["Margin"] = g["Profit"] / g["Sales"]
    g["SalesShare"] = g["Sales"] / g["Sales"].sum()
    return g.sort_values("Sales", ascending=False)

dump(agg_by(["Category"]).round(4), "t12_category")
dump(agg_by(["Category", "Sub-Category"]).round(4), "t13_subcategory")
dump(agg_by(["Region"]).round(4), "t14_region")
dump(agg_by(["State"]).round(4), "t15_state")
dump(agg_by(["Segment"]).round(4), "t16_segment")
dump(agg_by(["Discount Band"]).round(4), "t17_discount_band")
dump(agg_by(["Ship Mode"]).round(4), "t18_shipmode")
dump(agg_by(["Order Year"]).round(4), "t19_year")

prod = (df.groupby(["Product ID", "Product Name", "Category", "Sub-Category"]).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Qty=("Quantity", "sum"), avgDisc=("Discount", "mean")).reset_index())
dump(prod.nlargest(20, "Profit").round(4), "t20_top_product")
dump(prod.nsmallest(20, "Profit").round(4), "t21_loss_product")

cols = ["Sales", "Profit", "Quantity", "Discount", "Profit Margin", "Ship Days", "Unit Price"]
d = df[cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]).T
d["skew"] = df[cols].skew()
d["kurt"] = df[cols].kurt()
dump(d.reset_index().rename(columns={"index": "字段"}).round(4), "t22_describe")

g = (df.groupby("Discount Band", observed=True).agg(Rows=("Sales", "size"), Sales=("Sales", "sum"), Profit=("Profit", "sum")).reset_index())
g["Margin"] = g["Profit"] / g["Sales"]
g["ProfitShare"] = g["Profit"] / df["Profit"].sum()
dump(g.round(4), "t23_discount_loss")

pivot = (df.pivot_table(index="Order Year", columns="Discount Band", values="Profit", aggfunc="sum", observed=True) / df.pivot_table(index="Order Year", columns="Discount Band", values="Sales", aggfunc="sum", observed=True)).round(4)
pivot.reset_index().to_csv(os.path.join(TAB_DIR, "t24_year_discount_margin.csv"), index=False, encoding="utf-8-sig")
print("   t24_year_discount_margin.csv")

full = df[df["Discount"] == 0]
print("\n  核心发现：")
print(f"    全价行利润 {full['Profit'].sum()/1e4:,.1f} 万美元，占总额 {full['Profit'].sum()/profit*100:.0f}%")
print(f"    有折扣行合计利润 {(df[df['Discount'] > 0]['Profit'].sum())/1e4:,.1f} 万美元，折扣>20% 后利润率为负")

探索性分析
   t10_overview.csv  15 行
   t11_monthly.csv  48 行
   t12_category.csv  3 行
   t13_subcategory.csv  17 行
   t14_region.csv  4 行
   t15_state.csv  49 行
   t16_segment.csv  3 行
   t17_discount_band.csv  7 行
   t18_shipmode.csv  4 行
   t19_year.csv  4 行
   t20_top_product.csv  20 行
   t21_loss_product.csv  20 行
   t22_describe.csv  7 行
   t23_discount_loss.csv  7 行
   t24_year_discount_margin.csv

  核心发现：
    全价行利润 32.1 万美元，占总额 112%
    有折扣行合计利润 -3.5 万美元，折扣>20% 后利润率为负


本模块对应论文第四节的核心商业分析模型。代码共实现了 10 类高阶数据科学模型，涵盖趋势分析、帕累托/ABC分类、波士顿矩阵、RFM分层、同期群留存、K-Means聚类、SARIMA预测、Apriori关联规则、折扣弹性回归及漏斗转化。每种模型均会将其计算结果导出至 outputs/tables 目录下，这些结果表将直接作为后续自动化绘图脚本的输入源，同时也为论文中的商业洞察提供坚实的数据支撑。

In [2]:
import os
import itertools
import numpy as np
import pandas as pd

current_dir = os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, ".."))
CLEAN_DIR = os.path.join(base_dir, "data", "clean")
TAB_DIR = os.path.join(base_dir, "outputs", "tables")

ABC_P = 0.8
SEED = 42
FORECAST_HOLDOUT = 6
ANALYSIS_DATE = pd.Timestamp("2018-12-31")

def load():
    return pd.read_csv(os.path.join(CLEAN_DIR, "orders_clean.csv"), parse_dates=["Order Date", "Ship Date"])

def dump(df, name):
    df.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=False, encoding="utf-8-sig")
    return df

df = load()

# ---------------- M1 趋势与对比 ----------------
print("\n M1 趋势与对比")
m = (df.groupby("Order YM").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique")).reset_index())
m["Month"] = pd.PeriodIndex(m["Order YM"], freq="M").to_timestamp()
m["Margin"] = m["Profit"] / m["Sales"]
m["MoM"] = m["Sales"].pct_change()
m["YoY"] = m["Sales"].pct_change(12)
m["MA3"] = m["Sales"].rolling(3, center=True).mean()
m["MonthNo"] = m["Month"].dt.month
season = m.groupby("MonthNo")["Sales"].mean()
m["SeasonIdx"] = m["MonthNo"].map(season / season.mean())
dump(m.round(4), "t30_trend_monthly")
print(f"    · 销售旺季：{season.idxmax()} 月（季节指数 {season.max()/season.mean():.2f}）")

yc = df.groupby(["Order Year", "Category"])["Sales"].sum().unstack()
share = (yc.T / yc.sum(axis=1)).T.round(4)
share.reset_index().to_csv(os.path.join(TAB_DIR, "t31_structure_share.csv"), index=False, encoding="utf-8-sig")

# ---------------- M2 帕累托 / ABC ----------------
print("\n M2 帕累托 / ABC")
def abc(keys, value):
    g = df.groupby(keys)[value].sum().reset_index().sort_values(value, ascending=False)
    total = g.loc[g[value] > 0, value].sum()
    g["CumShare"] = g[value].clip(lower=0).cumsum() / total
    g["Rank"] = np.arange(1, len(g) + 1)
    g["Class"] = np.where(g["CumShare"] <= ABC_P, "A", np.where(g["CumShare"] <= 0.95, "B", "C"))
    return g

sub = abc(["Category", "Sub-Category"], "Profit")
dump(sub.round(4), "t32_abc_subcategory")
a = sub[sub["Class"] == "A"]
print(f"    · A 类子类 {len(a)} 个，贡献利润 {a['Profit'].sum()/sub['Profit'].sum()*100:.0f}%")
neg = sub[sub["Profit"] < 0]
print(f"    · 负利润子类 {len(neg)} 个：{list(neg['Sub-Category'])}，拖累 {neg['Profit'].sum()/1e4:.2f} 万美元")

# ---------------- M3 波士顿矩阵 ----------------
print("\n M3 波士顿矩阵")
g = (df.groupby(["Category", "Sub-Category"]).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Qty=("Quantity", "sum")).reset_index())
g["Share"] = g["Sales"] / g["Sales"].sum()
g["RelShare"] = g["Sales"] / g["Sales"].max()
y = df.groupby(["Sub-Category", "Order Year"])["Sales"].sum().unstack()
y["Growth"] = (y[2018] - y[2017]) / y[2017]
g = g.merge(y[["Growth"]].reset_index(), on="Sub-Category", how="left")
x_line, y_line = 0.35, (df[df["Order Year"] == 2018]["Sales"].sum() / df[df["Order Year"] == 2017]["Sales"].sum() - 1)
g["Quadrant"] = np.select(
    [(g["RelShare"] >= x_line) & (g["Growth"] >= y_line), (g["RelShare"] >= x_line) & (g["Growth"] < y_line), (g["RelShare"] < x_line) & (g["Growth"] >= y_line)],
    ["明星", "金牛", "问题"], default="瘦狗")
dump(g.round(4), "t34_bcg")
print(f"    · 四象限划分线：相对份额 {x_line}，增长率 {y_line*100:.1f}%")

# ---------------- M4 RFM 八分类 ----------------
print("\n M4 RFM 八分类")
tmp = df.sort_values("Order Date").copy()
region_mode = tmp.groupby("Customer ID")["Region"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[-1])
rfm = (tmp.groupby("Customer ID").agg(CustomerName=("Customer Name", "last"), Segment=("Segment", "last"), R=("Order Date", lambda s: (ANALYSIS_DATE - s.max()).days), F=("Order ID", "nunique"), M=("Sales", "sum"), Profit=("Profit", "sum")).reset_index())
rfm["Region"] = rfm["Customer ID"].map(region_mode)
rfm["Margin"] = rfm["Profit"] / rfm["M"]

r_mid, f_mid, m_mid = rfm["R"].median(), rfm["F"].median(), rfm["M"].median()
rfm["R_score"] = np.where(rfm["R"] <= r_mid, "高", "低")
rfm["F_score"] = np.where(rfm["F"] >= f_mid, "高", "低")
rfm["M_score"] = np.where(rfm["M"] >= m_mid, "高", "低")

naming = {("高", "高", "高"): "重要价值客户", ("低", "高", "高"): "重要唤回客户", ("高", "低", "高"): "重要发展客户", ("低", "低", "高"): "重要挽留客户", ("高", "高", "低"): "一般价值客户", ("低", "高", "低"): "一般保持客户", ("高", "低", "低"): "一般发展客户", ("低", "低", "低"): "一般挽留客户"}
rfm["RFM"] = [naming[(r, f, m)] for r, f, m in zip(rfm["R_score"], rfm["F_score"], rfm["M_score"])]
summary = (rfm.groupby("RFM").agg(客户数=("Customer ID", "count"), 销售额=("M", "sum"), 利润=("Profit", "sum"), 人均消费=("M", "mean")).reset_index())
summary["销售占比"] = summary["销售额"] / summary["销售额"].sum()
dump(rfm.round(4), "t35_rfm_detail")
dump(summary.round(4), "t36_rfm_summary")

# ---------------- M5 同期群留存 ----------------
print("\n M5 同期群留存")
d = df[["Customer ID", "Order Date"]].copy()
d["Order Month"] = d["Order Date"].dt.to_period("M")
first = d.groupby("Customer ID")["Order Month"].min().rename("Cohort")
d = d.join(first, on="Customer ID")
d["PeriodIdx"] = (d["Order Month"] - d["Cohort"]).apply(lambda x: x.n)
size = d.groupby("Cohort")["Customer ID"].nunique().rename("CohortSize")
active = (d.groupby(["Cohort", "PeriodIdx"])["Customer ID"].nunique().rename("Active").reset_index())
mat = active[active["PeriodIdx"] < 13].merge(size, on="Cohort")
mat["Retention"] = mat["Active"] / mat["CohortSize"]
pivot = mat.pivot_table(index="Cohort", columns="PeriodIdx", values="Retention", aggfunc="first")
pivot.index = [str(p) for p in pivot.index]
pivot.columns = [f"M{c}" for c in pivot.columns]
pivot.reset_index().to_csv(os.path.join(TAB_DIR, "t37_cohort_retention.csv"), index=False, encoding="utf-8-sig")
print(f"  [表] t37_cohort_retention.csv（{len(pivot)} 个群组）")

# ---------------- M7 SARIMA 预测 (跳过M6以节省时间，需要时可开启) ----------------
print("\n M7 SARIMA 预测（需等待几十秒）")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

s = (df.groupby("Order YM")["Sales"].sum())
s.index = pd.PeriodIndex(s.index, freq="M").to_timestamp()
s = s.asfreq("MS")
y_arr = s.values.astype(float)
n = len(y_arr)
h = FORECAST_HOLDOUT
tr, te = y_arr[:-h], y_arr[-h:]

def fit_sarima(train, steps_):
    mod = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12), enforce_stationarity=False, enforce_invertibility=False)
    res = mod.fit(disp=False)
    fc = res.get_forecast(steps=steps_)
    return res, fc.predicted_mean, fc.conf_int(alpha=0.05)

def fit_hw(train, steps_):
    mod = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12, initialization_method="estimated")
    res = mod.fit()
    pred = res.forecast(steps_)
    resid_std = np.std(res.resid)
    ci = np.column_stack([pred - 1.96 * np.sqrt(1 + np.arange(steps_)) * resid_std, pred + 1.96 * np.sqrt(1 + np.arange(steps_)) * resid_std])
    return res, np.asarray(pred), ci

def metrics(actual, pred):
    return np.mean(np.abs((actual - pred) / actual)) * 100, np.sqrt(np.mean((actual - pred) ** 2))

results = {}
for name, fn in [("SARIMA", fit_sarima), ("Holt-Winters", fit_hw)]:
    _, pred_te, _ = fn(tr, h)
    mape, rmse = metrics(te, pred_te)
    results[name] = dict(mape=mape, rmse=rmse, pred=pred_te, model=fn)
    print(f"    · {name} 回测：MAPE {mape:.2f}%，RMSE {rmse:,.0f}")

best = min(results, key=lambda k: results[k]["mape"])
print(f"    · 选定主模型：{best}")
_, fc_mean, fc_ci = results[best]["model"](y_arr, 6)
fut_idx = pd.date_range(s.index[-1] + pd.offsets.MonthBegin(1), periods=6, freq="MS")
out = pd.DataFrame({"date": list(s.index) + list(fut_idx), "actual": list(y_arr) + [np.nan]*6, "fitted": [np.nan]*n + list(fc_mean), "lower": [np.nan]*n + list(fc_ci[:, 0]), "upper": [np.nan]*n + list(fc_ci[:, 1]), "is_forecast": [0]*n + [1]*6})
dump(out.round(2), "t39_forecast")

# ---------------- M9 折扣弹性 ----------------
print("\n M9 折扣-利润弹性")
from sklearn.linear_model import LinearRegression
o = (df.groupby("Order ID").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Discount=("Discount", "max")).reset_index())
o["Margin"] = o["Profit"] / o["Sales"]
X = np.column_stack([o["Discount"], o["Discount"] ** 2])
lr = LinearRegression().fit(X, o["Margin"])
print(f"    · 回归 R²={lr.score(X, o['Margin']):.3f}，折扣的二次方系数 {lr.coef_[1]:.3f}")


 M1 趋势与对比
    · 销售旺季：11 月（季节指数 1.84）

 M2 帕累托 / ABC
    · A 类子类 6 个，贡献利润 81%
    · 负利润子类 3 个：['Supplies', 'Bookcases', 'Tables']，拖累 -2.24 万美元

 M3 波士顿矩阵
    · 四象限划分线：相对份额 0.35，增长率 20.4%

 M4 RFM 八分类

 M5 同期群留存
  [表] t37_cohort_retention.csv（42 个群组）

 M7 SARIMA 预测（需等待几十秒）
    · SARIMA 回测：MAPE 18.02%，RMSE 17,959
    · Holt-Winters 回测：MAPE 15.59%，RMSE 17,764
    · 选定主模型：Holt-Winters

 M9 折扣-利润弹性
    · 回归 R²=0.580，折扣的二次方系数 -1.027


In [4]:
import os
import itertools
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ================= 路径配置 =================
current_dir = os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, ".."))
CLEAN_DIR = os.path.join(base_dir, "data", "clean")
TAB_DIR = os.path.join(base_dir, "outputs", "tables")
os.makedirs(TAB_DIR, exist_ok=True)

ABC_P = 0.8
SEED = 42
FORECAST_HOLDOUT = 6
ANALYSIS_DATE = pd.Timestamp("2018-12-31")

def load():
    return pd.read_csv(os.path.join(CLEAN_DIR, "orders_clean.csv"), parse_dates=["Order Date", "Ship Date"])

def dump(df, name):
    df.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=False, encoding="utf-8-sig")
    return df

df = load()
print("数据加载完成，开始运行 10 个商业模型...")

#M1 趋势与对比
print("\n[M1] 趋势与对比")
m = (df.groupby("Order YM").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique")).reset_index())
m["Month"] = pd.PeriodIndex(m["Order YM"], freq="M").to_timestamp()
m["Margin"] = m["Profit"] / m["Sales"]
m["MoM"] = m["Sales"].pct_change()
m["YoY"] = m["Sales"].pct_change(12)
m["MA3"] = m["Sales"].rolling(3, center=True).mean()
m["MonthNo"] = m["Month"].dt.month
season = m.groupby("MonthNo")["Sales"].mean()
m["SeasonIdx"] = m["MonthNo"].map(season / season.mean())
dump(m.round(4), "t30_trend_monthly")
print(f"    销售旺季：{season.idxmax()} 月（季节指数 {season.max()/season.mean():.2f}）")

yc = df.groupby(["Order Year", "Category"])["Sales"].sum().unstack()
share = (yc.T / yc.sum(axis=1)).T.round(4)
dump(share.reset_index(), "t31_structure_share")

#M2 帕累托 / ABC
print("\n[M2] 帕累托 / ABC")
def abc(keys, value):
    g = df.groupby(keys)[value].sum().reset_index().sort_values(value, ascending=False)
    total = g.loc[g[value] > 0, value].sum()
    g["CumShare"] = g[value].clip(lower=0).cumsum() / total
    g["Rank"] = np.arange(1, len(g) + 1)
    g["Class"] = np.where(g["CumShare"] <= ABC_P, "A", np.where(g["CumShare"] <= 0.95, "B", "C"))
    return g

sub = abc(["Category", "Sub-Category"], "Profit")
dump(sub.round(4), "t32_abc_subcategory")
a = sub[sub["Class"] == "A"]
print(f"    A 类子类 {len(a)} 个，贡献利润 {a['Profit'].sum()/sub['Profit'].sum()*100:.0f}%")
neg = sub[sub["Profit"] < 0]
print(f"    负利润子类 {len(neg)} 个：{list(neg['Sub-Category'])}，拖累 {neg['Profit'].sum()/1e4:.2f} 万美元")

#M3 波士顿矩阵
print("\n[M3] 波士顿矩阵")
g = (df.groupby(["Category", "Sub-Category"]).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Qty=("Quantity", "sum")).reset_index())
g["Share"] = g["Sales"] / g["Sales"].sum()
g["RelShare"] = g["Sales"] / g["Sales"].max()
y = df.groupby(["Sub-Category", "Order Year"])["Sales"].sum().unstack()
y["Growth"] = (y[2018] - y[2017]) / y[2017]
g = g.merge(y[["Growth"]].reset_index(), on="Sub-Category", how="left")
x_line, y_line = 0.35, (df[df["Order Year"] == 2018]["Sales"].sum() / df[df["Order Year"] == 2017]["Sales"].sum() - 1)
g["Quadrant"] = np.select(
    [(g["RelShare"] >= x_line) & (g["Growth"] >= y_line), (g["RelShare"] >= x_line) & (g["Growth"] < y_line), (g["RelShare"] < x_line) & (g["Growth"] >= y_line)],
    ["明星", "金牛", "问题"], default="瘦狗")
dump(g.round(4), "t34_bcg")
print(f"    四象限划分线：相对份额 {x_line}，增长率 {y_line*100:.1f}%")

#M4 RFM 八分类
print("\n[M4] RFM 八分类")
tmp = df.sort_values("Order Date").copy()
region_mode = tmp.groupby("Customer ID")["Region"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[-1])
rfm = (tmp.groupby("Customer ID").agg(CustomerName=("Customer Name", "last"), Segment=("Segment", "last"), R=("Order Date", lambda s: (ANALYSIS_DATE - s.max()).days), F=("Order ID", "nunique"), M=("Sales", "sum"), Profit=("Profit", "sum")).reset_index())
rfm["Region"] = rfm["Customer ID"].map(region_mode)
rfm["Margin"] = rfm["Profit"] / rfm["M"]

r_mid, f_mid, m_mid = rfm["R"].median(), rfm["F"].median(), rfm["M"].median()
rfm["R_score"] = np.where(rfm["R"] <= r_mid, "高", "低")
rfm["F_score"] = np.where(rfm["F"] >= f_mid, "高", "低")
rfm["M_score"] = np.where(rfm["M"] >= m_mid, "高", "低")

naming = {("高", "高", "高"): "重要价值客户", ("低", "高", "高"): "重要唤回客户", ("高", "低", "高"): "重要发展客户", ("低", "低", "高"): "重要挽留客户", ("高", "高", "低"): "一般价值客户", ("低", "高", "低"): "一般保持客户", ("高", "低", "低"): "一般发展客户", ("低", "低", "低"): "一般挽留客户"}
rfm["RFM"] = [naming[(r, f, m)] for r, f, m in zip(rfm["R_score"], rfm["F_score"], rfm["M_score"])]
summary = (rfm.groupby("RFM").agg(客户数=("Customer ID", "count"), 销售额=("M", "sum"), 利润=("Profit", "sum"), 人均消费=("M", "mean")).reset_index())
summary["销售占比"] = summary["销售额"] / summary["销售额"].sum()
dump(rfm.round(4), "t35_rfm_detail")
dump(summary.round(4), "t36_rfm_summary")

#M5 同期群留存
print("\n[M5] 同期群留存")
d = df[["Customer ID", "Order Date"]].copy()
d["Order Month"] = d["Order Date"].dt.to_period("M")
first = d.groupby("Customer ID")["Order Month"].min().rename("Cohort")
d = d.join(first, on="Customer ID")
d["PeriodIdx"] = (d["Order Month"] - d["Cohort"]).apply(lambda x: x.n)
size = d.groupby("Cohort")["Customer ID"].nunique().rename("CohortSize")
active = (d.groupby(["Cohort", "PeriodIdx"])["Customer ID"].nunique().rename("Active").reset_index())
mat = active[active["PeriodIdx"] < 13].merge(size, on="Cohort")
mat["Retention"] = mat["Active"] / mat["CohortSize"]
pivot = mat.pivot_table(index="Cohort", columns="PeriodIdx", values="Retention", aggfunc="first")
pivot.index = [str(p) for p in pivot.index]
pivot.columns = [f"M{c}" for c in pivot.columns]
dump(pivot.reset_index(), "t37_cohort_retention")
print(f"    留存矩阵已生成（{len(pivot)} 个群组）")

#M6 K-Means 聚类 
print("\n[M6] K-Means 聚类")
X = rfm[["R", "F", "M"]].copy()
X["M"] = np.log1p(X["M"])
X["F"] = np.log1p(X["F"])
Z = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=4, random_state=SEED, n_init=10).fit(Z)
out = rfm.copy()
out["Cluster"] = km.labels_
prof = out.groupby("Cluster").agg(N=("Customer ID", "count"), R=("R", "mean"), F=("F", "mean"), M=("M", "mean"), Profit=("Profit", "sum")).reset_index()
rank = prof.sort_values("M")["Cluster"].tolist()
mp = {c: i for i, c in enumerate(rank)}
out["Cluster"] = out["Cluster"].map(mp)
prof["Cluster"] = prof["Cluster"].map(mp)
dump(out[["Customer ID", "CustomerName", "Segment", "R", "F", "M", "Cluster"]].round(3), "t38_kmeans_detail")
dump(prof.sort_values("Cluster").round(4), "t38b_kmeans_profile")
print(f"    K-Means 聚类完成，低价值簇包含 {int(prof.loc[3, 'N'])} 人")

#M7 SARIMA / HW 预测 
print("\n[M7] 时间序列预测 (SARIMA / HW)")
s = (df.groupby("Order YM")["Sales"].sum())
s.index = pd.PeriodIndex(s.index, freq="M").to_timestamp()
s = s.asfreq("MS")
y_arr = s.values.astype(float)
n = len(y_arr)
h = FORECAST_HOLDOUT
tr, te = y_arr[:-h], y_arr[-h:]

def fit_sarima(train, steps_):
    mod = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12), enforce_stationarity=False, enforce_invertibility=False)
    res = mod.fit(disp=False)
    fc = res.get_forecast(steps=steps_)
    return res, fc.predicted_mean, fc.conf_int(alpha=0.05)

def fit_hw(train, steps_):
    mod = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12, initialization_method="estimated")
    res = mod.fit()
    pred = res.forecast(steps_)
    resid_std = np.std(res.resid)
    ci = np.column_stack([pred - 1.96 * np.sqrt(1 + np.arange(steps_)) * resid_std, pred + 1.96 * np.sqrt(1 + np.arange(steps_)) * resid_std])
    return res, np.asarray(pred), ci

def metrics(actual, pred):
    return np.mean(np.abs((actual - pred) / actual)) * 100, np.sqrt(np.mean((actual - pred) ** 2))

results = {}
for name, fn in [("SARIMA", fit_sarima), ("Holt-Winters", fit_hw)]:
    _, pred_te, _ = fn(tr, h)
    mape, rmse = metrics(te, pred_te)
    results[name] = dict(mape=mape, rmse=rmse, pred=pred_te, model=fn)
    print(f"    {name} 回测：MAPE {mape:.2f}%，RMSE {rmse:,.0f}")

best = min(results, key=lambda k: results[k]["mape"])
print(f"    选定主模型：{best}")
_, fc_mean, fc_ci = results[best]["model"](y_arr, 6)
fut_idx = pd.date_range(s.index[-1] + pd.offsets.MonthBegin(1), periods=6, freq="MS")
out_fc = pd.DataFrame({"date": list(s.index) + list(fut_idx), "actual": list(y_arr) + [np.nan]*6, "fitted": [np.nan]*n + list(fc_mean), "lower": [np.nan]*n + list(fc_ci[:, 0]), "upper": [np.nan]*n + list(fc_ci[:, 1]), "is_forecast": [0]*n + [1]*6})
dump(out_fc.round(2), "t39_forecast")

#M8 Apriori 关联规则
print("\n[M8] Apriori 关联规则")
baskets = df.groupby("Order ID")["Sub-Category"].apply(frozenset).tolist()
n = len(baskets)
cnt = {}
for b in baskets:
    for it in b: cnt[frozenset([it])] = cnt.get(frozenset([it]), 0) + 1
min_sup = 0.008
freq = {k: v for k, v in cnt.items() if v / n >= min_sup}
allf = dict(freq)
k = 2
while k <= 3 and freq:
    prev = list(freq)
    cands = {a | b for a, b in itertools.combinations(prev, 2) if len(a | b) == k}
    freq = {}
    for c in cands:
        if any(frozenset(x) not in allf for x in itertools.combinations(c, k - 1)): continue
        m = sum(1 for b in baskets if c <= b)
        if m / n >= min_sup: freq[c] = m
    allf.update(freq)
    k += 1
rows = []
min_conf = 0.15
for items, m in allf.items():
    if len(items) < 2: continue
    for r in range(1, len(items)):
        for ante in itertools.combinations(items, r):
            a = frozenset(ante); c = items - a; sup_a = allf.get(a, 0) / n; conf = (m / n) / sup_a if sup_a else 0; sup_c = sum(1 for b in baskets if c <= b) / n
            if conf >= min_conf and sup_c: rows.append({"前项": " + ".join(sorted(a)), "后项": " + ".join(sorted(c)), "支持度": m / n, "置信度": conf, "提升度": conf / sup_c})
rules = pd.DataFrame(rows).sort_values("提升度", ascending=False).reset_index(drop=True)
dump(rules.round(4), "t42_association_rules")
print(f"    频繁项集 {len(allf)} 个，强规则 {len(rules)} 条")

#M9 折扣弹性
print("\n[M9] 折扣-利润弹性")
o = (df.groupby("Order ID").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Discount=("Discount", "max")).reset_index())
o["Margin"] = o["Profit"] / o["Sales"]
X = np.column_stack([o["Discount"], o["Discount"] ** 2])
lr = LinearRegression().fit(X, o["Margin"])
print(f"    回归 R²={lr.score(X, o['Margin']):.3f}，折扣的二次方系数 {lr.coef_[1]:.3f}")

#M10 折扣深度漏斗
print("\n[M10] 折扣深度漏斗")
o_funnel = df.groupby("Order ID").agg(md=("Discount", "max"), Sales=("Sales", "sum"), Profit=("Profit", "sum"))
funnel_rows = []
for lab, th in [("全部订单", -1), ("有折扣", 0), ("折扣>10%", 0.1), ("折扣>20%", 0.2), ("折扣>30%", 0.3)]:
    sub = o_funnel if th < 0 else o_funnel[o_funnel["md"] > th]
    funnel_rows.append({"层级": lab, "订单数": len(sub), "占比": len(sub) / len(o_funnel), "利润率": sub["Profit"].sum() / sub["Sales"].sum()})
fun = pd.DataFrame(funnel_rows)
dump(fun.round(4), "t45_funnel")
print(f"    漏斗计算完成，全部订单 {len(o_funnel)} 笔")

print("\n商业模型全部计算完毕！所有结果表已存入 outputs/tables/ 文件夹。")

数据加载完成，开始运行 10 个商业模型...

[M1] 趋势与对比
    销售旺季：11 月（季节指数 1.84）

[M2] 帕累托 / ABC
    A 类子类 6 个，贡献利润 81%
    负利润子类 3 个：['Supplies', 'Bookcases', 'Tables']，拖累 -2.24 万美元

[M3] 波士顿矩阵
    四象限划分线：相对份额 0.35，增长率 20.4%

[M4] RFM 八分类

[M5] 同期群留存
    留存矩阵已生成（42 个群组）

[M6] K-Means 聚类


D:\anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


    K-Means 聚类完成，低价值簇包含 57 人

[M7] 时间序列预测 (SARIMA / HW)
    SARIMA 回测：MAPE 18.02%，RMSE 17,959
    Holt-Winters 回测：MAPE 15.59%，RMSE 17,764
    选定主模型：Holt-Winters

[M8] Apriori 关联规则
    频繁项集 87 个，强规则 82 条

[M9] 折扣-利润弹性
    回归 R²=0.580，折扣的二次方系数 -1.027

[M10] 折扣深度漏斗
    漏斗计算完成，全部订单 5009 笔

商业模型全部计算完毕！所有结果表已存入 outputs/tables/ 文件夹。
